# GEO tokens: does a short fine-tune close the gap?

Public data and models only. ImageNet-R split per class 70/30; the 30% is never trained on. A public ViT-B/16 is scored before and after one epoch of training on shrink-rule token mosaics at random budgets (64-256), against plain grids.

In [ ]:
# setup
import os, re, json, time, random, math
import numpy as np
import torch
import torch.nn.functional as F
import timm
from PIL import Image
from torch.utils.data import Dataset, DataLoader
dev = "cuda"
torch.backends.cuda.matmul.allow_tf32 = True
print("torch", torch.__version__, "timm", timm.__version__, torch.cuda.get_device_name(0))
SEED = 0
BUDGETS = (64, 100, 130, 160, 196, 256)        # training budgets (shrink rule)
EVAL_KEYS = (64, 100, 130, 160, 256, "g64", "g256")
N_TEST = 3000                                  # held-out images scored before and after
EPOCHS, BS, LR, WARMUP = 1, 32, 1e-5, 50
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

In [ ]:
# tokenizers (tokens_core.py)
"""Token layouts at an exact budget, for a 256x256 image with 16px leaves.

Every method gives each splittable node (256, 128, 64, 32 px) a "loss" score, then a
greedy quadtree splits the highest-loss node until the budget is reached. Each split
adds 3 tokens, so budgets are 1 + 3k (64, 100, 130, 160, 196, 256).

  geo_uniform : max pixel deviation from the node mean, YCbCr, chroma counted at half
                weight (canPrune's rule as a score: the q at which the node would merge)
  geo_shrink  : max pixel deviation from its own s/16 cell mean, same weighting
                (shrink-safe as a score: how much is lost shrinking the node to 16x16)
  pixel_blur  : Quadformer's baseline: sqrt(mean squared error of shrink-and-blow-up * s), RGB
  entropy     : 256-bin grey-level entropy (APT-style)
  random      : random score (sanity floor)
"""
import heapq
import numpy as np

IMG, LEAF = 256, 16
SIZES = (256, 128, 64, 32)          # splittable node sizes


def ycc(rgb01):
    r, g, b = rgb01[..., 0] * 255, rgb01[..., 1] * 255, rgb01[..., 2] * 255
    y = 0.299 * r + 0.587 * g + 0.114 * b
    cb = 128 - 0.168736 * r - 0.331264 * g + 0.5 * b
    cr = 128 + 0.5 * r - 0.418688 * g - 0.081312 * b
    return np.stack([y, cb, cr], -1)


def _blocks(a, s):
    """(H,W,C) -> (n, n, s, s, C) blocks of size s."""
    n = a.shape[0] // s
    return a.reshape(n, s, n, s, -1).transpose(0, 2, 1, 3, 4)


def _cell_mean(a, s):
    """Each pixel replaced by the mean of its s/16 cell inside s-blocks == shrink to 16 and blow up."""
    f = s // LEAF
    if f == 1:
        return a
    n = a.shape[0] // f
    m = a.reshape(n, f, n, f, -1).mean(axis=(1, 3))
    return np.repeat(np.repeat(m, f, 0), f, 1)


def scores(rgb01, method, rng=None):
    """dict size -> (n, n) loss array for every splittable node."""
    out = {}
    Y = ycc(rgb01)
    W = np.array([1.0, 0.5, 0.5])                       # chroma at half weight (canPrune: 2x tolerance)
    grey = np.clip(Y[..., 0], 0, 255).astype(np.int64)
    for s in SIZES:
        n = IMG // s
        if method == "geo_uniform":
            b = _blocks(Y, s)
            dev = np.abs(b - b.mean(axis=(2, 3), keepdims=True)) * W
            out[s] = dev.max(axis=(2, 3, 4))
        elif method == "geo_shrink":
            dev = np.abs(Y - _cell_mean(Y, s)) * W
            out[s] = _blocks(dev, s).max(axis=(2, 3, 4))
        elif method == "pixel_blur":
            se = (rgb01 - _cell_mean(rgb01, s)) ** 2
            mse = _blocks(se, s).mean(axis=(2, 3, 4))
            out[s] = np.sqrt(mse * s)
        elif method == "entropy":
            g = _blocks(grey[..., None], s)[..., 0]
            e = np.zeros((n, n))
            for i in range(n):
                for j in range(n):
                    h = np.bincount(g[i, j].ravel(), minlength=256).astype(float)
                    p = h[h > 0] / h.sum()
                    e[i, j] = -(p * np.log2(p)).sum()
            out[s] = e
        elif method == "random":
            out[s] = rng.random((n, n))
        else:
            raise ValueError(method)
    return out


def layout(sc, budget):
    """Greedy split: highest loss first. Returns list of (y, x, size)."""
    assert (budget - 1) % 3 == 0 and 1 <= budget <= (IMG // LEAF) ** 2, budget
    heap = [(-sc[IMG][0, 0], 0, 0, IMG)]
    leaves = []
    count = 1
    while count < budget:
        _, y, x, s = heapq.heappop(heap)
        h = s // 2
        for dy in (0, h):
            for dx in (0, h):
                yy, xx = y + dy, x + dx
                if h == LEAF:
                    leaves.append((yy, xx, h))
                else:
                    heapq.heappush(heap, (-sc[h][yy // h, xx // h], yy, xx, h))
        count += 3
        while heap and heap[0][3] == LEAF:      # never happens (leaves kept apart), safety
            leaves.append(heapq.heappop(heap)[1:])
    toks = leaves + [(y, x, s) for _, y, x, s in heap]
    assert len(toks) == budget and sum(s * s for _, _, s in toks) == IMG * IMG
    return toks


def grid(s):
    return [(y, x, s) for y in range(0, IMG, s) for x in range(0, IMG, s)]

In [ ]:
# data split (per class, 70% train / 30% test; test images are never trained on)
roots = [r for r, d, f in os.walk("/kaggle/input") if re.fullmatch(r"n\d{8}", os.path.basename(r))]
data_root = os.path.dirname(roots[0])
wnid_dirs = sorted(d for d in os.listdir(data_root) if re.fullmatch(r"n\d{8}", d))
rng = random.Random(SEED)
train_items, test_items = [], []
for ci, w in enumerate(wnid_dirs):
    fs = sorted(os.listdir(os.path.join(data_root, w)))
    rng.shuffle(fs)
    k = int(0.7 * len(fs))
    train_items += [(os.path.join(data_root, w, f), ci) for f in fs[:k]]
    test_items += [(os.path.join(data_root, w, f), ci) for f in fs[k:]]
rng.shuffle(train_items); rng.shuffle(test_items)
test_eval = test_items[:N_TEST]
assert not set(p for p, _ in train_items) & set(p for p, _ in test_items)
print(len(wnid_dirs), "classes |", len(train_items), "train |", len(test_items), "test |", len(test_eval), "scored")
in1k = list(timm.data.ImageNetInfo().label_names())
sub_idx = torch.tensor([in1k.index(w) for w in wnid_dirs], device=dev)

In [ ]:
# model and the mixed-token path
model = timm.create_model("vit_base_patch16_224.augreg2_in21k_ft_in1k", pretrained=True, img_size=256).to(dev)
cfg = timm.data.resolve_model_data_config(model)
MEAN = torch.tensor(cfg["mean"], device=dev).view(1, 3, 1, 1)
STD = torch.tensor(cfg["std"], device=dev).view(1, 3, 1, 1)
assert getattr(model, "num_reg_tokens", 0) == 0 and not model.no_embed_class


def load(path):
    im = Image.open(path).convert("RGB")
    w, h = im.size
    k = 256 / min(w, h)
    im = im.resize((max(256, round(w * k)), max(256, round(h * k))), Image.BICUBIC)
    w, h = im.size
    im = im.crop(((w - 256) // 2, (h - 256) // 2, (w - 256) // 2 + 256, (h - 256) // 2 + 256))
    return np.asarray(im, dtype=np.float64) / 255.0


class DS(Dataset):
    """Returns the image, its class, and the shrink-rule layout at every budget plus two plain grids."""
    def __init__(self, items, train):
        self.items, self.train = items, train
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        p, c = self.items[i]
        a = load(p)
        if self.train and random.random() < 0.5:
            a = a[:, ::-1].copy()
        sc = scores(a, "geo_shrink")
        lays = {b: layout(sc, b) for b in BUDGETS}
        lays["g64"], lays["g256"] = grid(32), grid(16)
        return torch.from_numpy(a.astype(np.float32)).permute(2, 0, 1), c, lays


def collate(batch):
    return torch.stack([b[0] for b in batch]), torch.tensor([b[1] for b in batch]), [b[2] for b in batch]


def embed_batch(imgs, lays):
    """imgs (N,3,256,256) in [0,1]; lays: one list of (y,x,s) per image, same length. -> (N, B, D).
    Each token is shrunk to 16x16 (area), embedded with the patch embedding, and given the mean of
    the grid position embeddings it covers. Differentiable, so the model can learn mixed sizes."""
    x = (imgs - MEAN) / STD
    P = model.pos_embed[0, 1:].reshape(16, 16, -1).permute(2, 0, 1)[None]
    tiles, pos = {}, {}
    for f in (1, 2, 4, 8, 16):
        xf = F.avg_pool2d(x, f) if f > 1 else x
        tiles[f] = xf.unfold(2, 16, 16).unfold(3, 16, 16).permute(0, 2, 3, 1, 4, 5)   # N,n,n,3,16,16
        pos[f] = (F.avg_pool2d(P, f) if f > 1 else P)[0]                               # D,n,n
    pats, poss = [], []
    for i, L in enumerate(lays):
        groups = {}
        for y, xx, s in L:
            groups.setdefault(s // 16, []).append((y // s, xx // s))
        for f, ij in groups.items():
            ij = torch.tensor(ij, device=x.device)
            pats.append(tiles[f][i, ij[:, 0], ij[:, 1]])
            poss.append(pos[f][:, ij[:, 0], ij[:, 1]].T)
    e = model.patch_embed.norm(model.patch_embed.proj(torch.cat(pats)).flatten(1)) + torch.cat(poss)
    return e.view(len(lays), len(lays[0]), -1)


def classify(seqs):
    cls = (model.cls_token[0] + model.pos_embed[0, :1]).expand(seqs.shape[0], 1, -1)
    x = torch.cat([cls, seqs], 1)
    x = model.norm(model.blocks(model.norm_pre(model.patch_drop(x))))
    return model.forward_head(x)


# sanity: the full grid through this path must equal the model's own forward
model.eval()
with torch.no_grad():
    imgs, labels, lays = collate([DS(test_eval[:4], False)[i] for i in range(4)])
    imgs = imgs.to(dev)
    ours = classify(embed_batch(imgs, [L["g256"] for L in lays]))
    ref = model((imgs - MEAN) / STD)
    d = float((ours - ref).abs().max())
print("sanity max |logit diff| full grid:", d)
assert d < 1e-3

In [ ]:
# evaluate before training
@torch.no_grad()
def evaluate(tag):
    model.eval()
    corr = {k: 0 for k in EVAL_KEYS}; n = 0; t0 = time.time()
    dl = DataLoader(DS(test_eval, False), batch_size=64, num_workers=4, collate_fn=collate)
    for imgs, labels, lays in dl:
        imgs, labels = imgs.to(dev, non_blocking=True), labels.to(dev)
        with torch.autocast("cuda", dtype=torch.float16):
            for k in EVAL_KEYS:
                pred = classify(embed_batch(imgs, [L[k] for L in lays]))[:, sub_idx].argmax(1)
                corr[k] += int((pred == labels).sum())
        n += len(labels)
    acc = {str(k): round(100 * corr[k] / n, 2) for k in EVAL_KEYS}
    print(f"{tag}: n={n} in {time.time() - t0:.0f}s  " + "  ".join(f"{k}={v}" for k, v in acc.items()), flush=True)
    return acc

results = {"before": evaluate("BEFORE (zero-shot)")}

In [ ]:
# fine-tune 1 epoch, random token budget per batch
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.05)
dl = DataLoader(DS(train_items, True), batch_size=BS, shuffle=True, num_workers=4,
                collate_fn=collate, drop_last=True, persistent_workers=False)
total = len(dl) * EPOCHS
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda s: min(1.0, (s + 1) / WARMUP) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / total))))
scaler = torch.cuda.amp.GradScaler()
t0 = time.time(); step = 0
for ep in range(EPOCHS):
    model.train()
    for imgs, labels, lays in dl:
        b = random.choice(BUDGETS)
        imgs, labels = imgs.to(dev, non_blocking=True), labels.to(dev)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = classify(embed_batch(imgs, [L[b] for L in lays]))[:, sub_idx]
            loss = F.cross_entropy(logits.float(), labels)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step(); step += 1
        if step % 50 == 0 or step == 1:
            print(f"step {step}/{total}  budget {b}  loss {loss.item():.3f}  {time.time() - t0:.0f}s", flush=True)

In [ ]:
# evaluate after, save
results["after"] = evaluate("AFTER (1 epoch, mixed tokens)")
results["config"] = dict(seed=SEED, train=len(train_items), scored=len(test_eval), epochs=EPOCHS, bs=BS, lr=LR,
                         budgets=BUDGETS, model="vit_base_patch16_224.augreg2_in21k_ft_in1k @256", rule="geo_shrink")
json.dump(results, open("/kaggle/working/finetune_results.json", "w"), indent=1)
print("\nTop-1 on held-out ImageNet-R (200 classes), shrink-rule tokens; g64/g256 = plain grid")
print(f"{'tokens':<8}{'before':>9}{'after':>9}")
for k in EVAL_KEYS:
    print(f"{str(k):<8}{results['before'][str(k)]:>8.1f}%{results['after'][str(k)]:>8.1f}%")
print("RESULTS_JSON " + json.dumps(results))